In [1]:
from openai import OpenAI
openai_client = OpenAI()

# Search index

In [2]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [3]:
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 85},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [4]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [5]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [6]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [7]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [16]:
search_results = index.search(
    "I just discovered the course. Can I join now?",
    num_results=5,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"}
)

[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

In [9]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [10]:
search("What happens if I miss the homework deadline?")

[{'id': '7d3a2e3505',
  'course': 'llm-zoomcamp',
  'section': 'Module 1 Homework',
  'question': 'What should I do if homework questions feel unclear?',
  'answer': 'First read the GitHub homework instructions, not only the course platform page.\n\nFor Module 1 in the 2026 cohort, start here:\n\n- [Module 1 cohort materials](https://github.com/DataTalksClub/llm-zoomcamp/tree/main/cohorts/2026/01-agentic-rag)\n- [Module 1 homework instructions](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/01-agentic-rag/homework.md)\n\nThe homework follows the lesson workflow, but usually uses a different dataset or asks you to apply the same idea in a slightly different way.\n\nIf it is still unclear, ask in Slack and include the code or command output as text, not a screenshot. Follow the [asking questions guidelines](https://datatalks.club/docs/courses/zoomcamp-logistics/asking-questions/).'},
 {'id': 'fe8fed31e6',
  'course': 'llm-zoomcamp',
  'section': 'Module 1 Homework',

# Building the prompt

In [11]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [40]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [41]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [42]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [43]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [44]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [46]:
response
# This is a Pydantic object and the output lives under "output"

Response(id='resp_0a836a94a9359a96006a3be6bdcd1881a2a2116dc1f44c7676', created_at=1782310589.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0a836a94a9359a96006a3be6bebfd881a2a0e4b51e8cad6558', content=[ResponseOutputText(annotations=[], text='Yes — you can still join now and start learning.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=1782310591.0, conversation=None, max_output_tokens=None, max_tool_calls=None, moderation=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_retention='24h', reasoning=Reasoning(context='current_turn', effort='none', generate_su

In [47]:
response.output

[ResponseOutputMessage(id='msg_0a836a94a9359a96006a3be6bebfd881a2a0e4b51e8cad6558', content=[ResponseOutputText(annotations=[], text='Yes — you can still join now and start learning.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]

In [48]:
response.output[0].content[0].text

'Yes — you can still join now and start learning.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open.'

In [49]:
response.output_text
# Shortcut

'Yes — you can still join now and start learning.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open.'

In [50]:
response.usage

ResponseUsage(input_tokens=480, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=41, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=521)

In [51]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.0005445000000000001

In [52]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
)

In [53]:
response.output_text


'Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still open.'

In [54]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [55]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [56]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join. If you want a certificate, you need to submit your project while submissions are still open.


In [57]:
rag("How do I get a certificate?")

'To get a certificate, you need to finish the course with a **live cohort** and complete the **Capstone project**.\n\nA few important points:\n- **Self-paced mode does not include certificates.**\n- You must also **peer-review 3 capstone projects** after submitting your own project.\n- If you missed the first homework, that’s okay — **homework is not mandatory** for the certificate.\n\nIf you want, I can also explain how to make sure your **certificate name** is correct.'

In [59]:
print(response.model_dump_json(indent = 2))

{
  "id": "resp_088aa637be3c0889006a3be6f68fcc819ca4f0acf8a7797615",
  "created_at": 1782310646.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5.4-mini-2026-03-17",
  "object": "response",
  "output": [
    {
      "id": "msg_088aa637be3c0889006a3be6f723ac819c9c1edd036ebe73d3",
      "content": [
        {
          "annotations": [],
          "text": "Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still open.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": "final_answer"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 0.98,
  "background": false,
  "completed_at": 1782310647.0,
  "conversation": null,
  "max_output_tokens": null,
  "max_tool_calls": null,
  "moderation": nul

# Helper functions

In [63]:
# Created under ingest.py and rag_helper.py

# Stuff with sqlite

In [2]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [3]:
from rag_helper import RAGBase
from openai import OpenAI

openai_client = OpenAI()

In [4]:
assistant = RAGBase(
    index=sqlite_index, #just swap minsearch index with this
    llm_client=openai_client,
)

# Defining a tool

In [62]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return sqlite_index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [9]:
search_tool = {
    "type": "function",
    # name needs to match the name of the function?
    "name": "search",
    # this is the most important field. LLM will use this information to decide when to call
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [7]:
# without search tool
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes—usually you can join if the course is still open and has available spots.\n\nA few things to check:\n- **Enrollment deadline**: Has registration closed?\n- **Prerequisites**: Do you meet any required background or skills?\n- **Capacity**: Is there still room in the class?\n- **Access method**: Some courses require approval from the instructor or admin.\n\nIf you want, send me the **course name/link** and I can help you figure out whether you’re likely eligible and what to do next.'

In [11]:
# with search tool
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool]
)

response.output
# This returns the decision of LLM to call the search tool, with arguments

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course late discovered the course join it enrollment deadline"}', call_id='call_XHfzWfZQr4hrBQJYRMWPfUXN', name='search', type='function_call', id='fc_02d822b630474e04006a3c6e63d158819db3ec99d8d73242c3', namespace=None, status='completed')]

In [12]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2) # this is the stuff search returns


In [16]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]

In [17]:
messages.extend(response.output)
# We add to the history the decision of LLM to use the search tool

In [18]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course late discovered the course join it enrollment deadline"}', call_id='call_XHfzWfZQr4hrBQJYRMWPfUXN', name='search', type='function_call', id='fc_02d822b630474e04006a3c6e63d158819db3ec99d8d73242c3', namespace=None, status='completed')]

In [19]:
messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})
# we also add the result of the search along with the call id

In [20]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course late discovered the course join it enrollment deadline"}', call_id='call_XHfzWfZQr4hrBQJYRMWPfUXN', name='search', type='function_call', id='fc_02d822b630474e04006a3c6e63d158819db3ec99d8d73242c3', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_XHfzWfZQr4hrBQJYRMWPfUXN',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and g

In [21]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course and start learning.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.'

In [22]:
response

Response(id='resp_02d822b630474e04006a3c7173ddd8819d88d12d1dfae36606', created_at=1782346099.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_02d822b630474e04006a3c71748cf4819da447df59d07718a9', content=[ResponseOutputText(annotations=[], text='Yes — you can still join the course and start learning.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='search', parameters={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query text to look up in the course FAQ.'}}, 'required': ['query'], 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Search the FAQ d

In [23]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(774, 34)

In [24]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


# Agent

In [25]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [26]:
def make_call(call):
    args = json.loads(call.arguments) # arguments -> dict

    if call.name == "search": # can be expanded as more tools come
        result = search(**args)

    result_json = json.dumps(result, indent=2) # result -> json

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [27]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

In [28]:
# function call loop
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False: # done if the response did not involve a function call
        break

iteration #1...
function_call: search {"query":"join course discovered course can I join late enrollment registration"}
function_call: search {"query":"late enrollment course join after start discovered the course"}
function_call: search {"query":"course enrollment can I join if I just discovered the course"}
iteration #2...
ASSISTANT:
Yes — you can still join the course and start learning.

A couple of notes:
- You don’t need any special confirmation to begin.
- If you want a certificate, you’ll need to submit your project while submissions are still open.

If you’d like, I can also tell you how to start the course workflow or what the deadlines look like. Any other areas you want to explore?


In [29]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages, # as the loop goes, this will expand
            tools=[search_tool]
        )

        messages.extend(response.output) # add the response to the history

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output) # add call output to the history
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [31]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run Ollama local install start serve FAQ"}
function_call: search {"query":"Ollama local run model pull serve localhost FAQ"}
function_call: search {"query":"install Ollama locally macOS Windows Linux FAQ"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - **macOS**: install the `.pkg`
   - **Windows**: install the `.msi`
   - **Linux**:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This downloads the model, starts it locally, and opens a chat-style prompt.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response from the Ollama server.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',


'To run Ollama locally:\n\n1. **Install Ollama**\n   - Go to: https://ollama.com/download\n   - **macOS**: install the `.pkg`\n   - **Windows**: install the `.msi`\n   - **Linux**:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model, starts it locally, and opens a chat-style prompt.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response from the Ollama server.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection issue, restarting the server can help:\n```bash\nnohup ollama serve > nohup.out 2>&1 &\n```\n\nIf you want, I can

## Multiple searches

In [34]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered late enroll can I join FAQ"}
iteration #2...
function_call: search {"query":"project submission still accepting submissions certificate join course late FAQ acceptance submitting project peer review"}
iteration #3...
ASSISTANT:
Yes, you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still being accepted. Also, certificates are only available for the live cohort, not self-paced study.

If you want, I can also help you figure out whether you can still get a certificate or what the project/peer-review requirements are.


'Yes, you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still being accepted. Also, certificates are only available for the live cohort, not self-paced study.\n\nIf you want, I can also help you figure out whether you can still get a certificate or what the project/peer-review requirements are.'

## Light guardrails against off topic stuff

In [36]:

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"gambit chess queen's gambit opening"}
iteration #3...
ASSISTANT:
I couldn’t find anything in the course FAQ about “queen gambit,” so it looks like this is off-topic for the course.

If you meant **the Queen’s Gambit in chess**, I can’t answer that here unless it appears in the course FAQ. If you want, I can still help you look for a course-related question instead.

Are there other areas you want to explore?


'I couldn’t find anything in the course FAQ about “queen gambit,” so it looks like this is off-topic for the course.\n\nIf you meant **the Queen’s Gambit in chess**, I can’t answer that here unless it appears in the course FAQ. If you want, I can still help you look for a course-related question instead.\n\nAre there other areas you want to explore?'

# ToyAIKit

In [37]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [67]:
search_tool

{'type': 'function',
 'name': 'search',
 'description': 'Search the FAQ database for entries matching the given query.',
 'parameters': {'type': 'object',
  'properties': {'query': {'type': 'string',
    'description': 'Search query text to look up in the course FAQ.'}},
  'required': ['query'],
  'additionalProperties': False}}

In [38]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [41]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'Search query text to look up in the course FAQ.'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [42]:
agent_tools = Tools()

In [43]:
agent_tools.get_tools()

[]

In [59]:
def search(query) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [60]:
agent_tools = Tools()
agent_tools.add_tool(search) # if the function has types and docstring, 
                             # schema is generated automatically
                             # Seems to generate schema either way, just without descr?

In [68]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

# this is the agent
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [73]:
result = runner.loop(
    prompt="How do I run Olbama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [75]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olbama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olbama run locally FAQ"}', call_id='call_93U7yl3mIrLUrzdys4tN

In [ ]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages, # this passes the history
    callback=callback,
)